### Chapter 7.2 - Convolutions for Images

This notebook makes the image convolution operation concrete using tiny matrices before mapping it to PyTorch layers.


In [ ]:
import torch
from torch import nn


### 7.2.1 The Cross-Correlation Operation

#### 1. Intuition

Cross-correlation slides a kernel over an input and computes a weighted sum at each valid location.

Valid location means the whole kernel fits inside the input without going outside the border.

#### 2. Why this exists

This operation turns local image patches into feature responses, which is the core computation of a convolutional layer.


#### 3. Examples

Implement two-dimensional cross-correlation from scratch.


In [ ]:
def corr2d(X, K):
    h, w = len(K), len(K[0])
    out = []
    for i in range(len(X) - h + 1):
        row = []
        for j in range(len(X[0]) - w + 1):
            row.append(sum(X[i+a][j+b] * K[a][b] for a in range(h) for b in range(w)))
        out.append(row)
    return out


Apply it to a tiny image.


In [ ]:
X = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]
K = [[1, 0], [0, -1]]
corr2d(X, K)


#### 4. Step-by-step breakdown

`corr2d` reads the kernel height and width.

The outer loops choose top-left window positions.

The inner expression multiplies matching input and kernel entries.

Each window produces one output value.

#### 5. Connection to ML systems

PyTorch uses highly optimized kernels for this same mathematical pattern.

#### 6. Common confusion points

- Cross-correlation does not flip the kernel.
- Output size shrinks when no padding is used.
- Every output value sees only one local window.
- Tiny manual code is for learning, not speed.


### 7.2.2 Convolutional Layers

#### 1. Intuition

A convolutional layer stores one or more learnable kernels.

During training, gradient descent changes the kernel values so useful patterns produce useful feature maps.

#### 2. Why this exists

Manual kernels are good for explanation, but a model needs trainable kernels that update from data.


#### 3. Examples

Create a minimal PyTorch convolutional layer.


In [ ]:
conv = nn.Conv2d(1, 1, kernel_size=2, bias=False)
X = torch.arange(9, dtype=torch.float32).reshape(1, 1, 3, 3)
Y = conv(X)
Y.shape


Inspect the stored kernel shape.


In [ ]:
conv.weight.shape


#### 4. Step-by-step breakdown

`1, 1` means one input channel and one output channel.

`kernel_size=2` means a 2 by 2 local window.

The input shape is batch, channels, height, width.

The output contains one learned response map.

#### 5. Connection to ML systems

`nn.Conv2d` is the standard PyTorch module for two-dimensional image convolution.

#### 6. Common confusion points

- The batch dimension is separate from channels.
- The layer stores parameters in `weight`.
- Kernel values start from initialization, not from hand-picked edge filters.
- Bias can be disabled for simpler examples.


### 7.2.3 Object Edge Detection in Images

#### 1. Intuition

An edge is a sharp change in pixel values.

A simple vertical-edge kernel can respond strongly when the left side of a window differs from the right side.

#### 2. Why this exists

Edges are useful low-level visual features. Early CNN layers often learn edge-like detectors automatically.


#### 3. Examples

Detect a vertical edge in a tiny image.


In [ ]:
X = [[1, 1, 0, 0],
     [1, 1, 0, 0],
     [1, 1, 0, 0]]
K = [[1, -1]]
corr2d(X, K)


#### 4. Step-by-step breakdown

The image has bright values on the left and dark values on the right.

The kernel subtracts the right pixel from the left pixel.

Large positive responses appear where brightness drops.

This is a handcrafted feature detector.

#### 5. Connection to ML systems

CNNs learn many such detectors from data instead of requiring humans to design every kernel.

#### 6. Common confusion points

- Edge detection is only an example, not all a CNN does.
- The sign of the response depends on kernel direction.
- Horizontal and vertical edges need different kernels.
- Learned filters can become more complex than simple edges.


### 7.2.4 Learning a Kernel

#### 1. Intuition

Learning a kernel means changing its values so the produced feature map matches a target behavior.

The kernel is just a small parameter tensor, so ordinary gradient descent can update it.

#### 2. Why this exists

Hand-designed kernels do not scale to real tasks. Learning lets the model discover useful local patterns from examples.


#### 3. Examples

Set up a tiny learnable kernel and loss.


In [ ]:
X = torch.tensor([[[[1., 1., 0., 0.]]]])
target = torch.tensor([[[[0., 1., 0.]]]])
conv = nn.Conv2d(1, 1, (1, 2), bias=False)
pred = conv(X)
loss = ((pred - target) ** 2).mean()
loss.backward()
conv.weight.grad


#### 4. Step-by-step breakdown

`conv.weight` is the learnable kernel.

`pred` is the feature map produced by the current kernel.

The loss measures mismatch with the target response.

`backward()` computes gradients for changing the kernel.

#### 5. Connection to ML systems

Training CNNs uses the same autograd and optimizer machinery as MLPs; only the layer computation changes.

#### 6. Common confusion points

- A kernel is learned through gradients.
- The target here is artificial to keep the example tiny.
- Real training uses many images and labels.
- Gradients say how to change weights, not the final values directly.


### 7.2.5 Cross-Correlation and Convolution

#### 1. Intuition

Mathematical convolution flips the kernel before sliding it.

Deep learning libraries usually implement cross-correlation but call the layer convolution because the learnable kernel can adapt either way.

#### 2. Why this exists

This naming mismatch appears in almost every CNN library and textbook, so it should be understood early.


#### 3. Examples

Show a one-dimensional flip.


In [ ]:
kernel = [1, 2, 3]
flipped = list(reversed(kernel))
kernel, flipped


Learnable kernels make the naming issue harmless.


In [ ]:
message = "A learned kernel can store either orientation."
message


#### 4. Step-by-step breakdown

A true convolution would use the flipped kernel.

Cross-correlation uses the kernel as written.

If the kernel is learned, the model can learn the needed orientation.

That is why `Conv2d` remains the standard name.

#### 5. Connection to ML systems

When reading papers, assume `convolution` in neural networks usually means the deep-learning convention unless stated otherwise.

#### 6. Common confusion points

- The distinction matters for manual math.
- It usually does not matter for learned CNN layers.
- PyTorch uses cross-correlation behavior.
- Be precise when implementing from scratch.


### 7.2.6 Feature Map and Receptive Field

#### 1. Intuition

A feature map is the output grid produced by a kernel.

A receptive field is the region of the input that can affect one output value.

#### 2. Why this exists

These terms describe what information each hidden value contains and how local evidence grows through layers.


#### 3. Examples

A 2 by 2 kernel sees four input positions.


In [ ]:
kernel_height, kernel_width = 2, 2
receptive_field = kernel_height * kernel_width
receptive_field


A feature map stores one response per window.


In [ ]:
input_size = 4
kernel_size = 2
output_size = input_size - kernel_size + 1
output_size


#### 4. Step-by-step breakdown

The receptive field area is the kernel area for one layer.

With no padding and stride 1, a 4-wide input becomes a 3-wide output.

Each output location summarizes a different local window.

Deeper layers can see larger input regions indirectly.

#### 5. Connection to ML systems

Architecture design often controls receptive field size through kernel size, depth, stride, and pooling.

#### 6. Common confusion points

- Feature map means output activations, not the original image.
- Receptive field grows across layers.
- A larger receptive field is not always better.
- Border handling affects which input positions are seen.


### 7.2.7 Summary

#### 1. Intuition

Image convolution turns small local windows into feature responses.

Kernels can be handcrafted for explanation or learned as model parameters.

#### 2. Why this exists

The operation gives CNNs their core ability to detect reusable local patterns.


#### 3. Examples

Summarize the core vocabulary.


In [ ]:
terms = {
    "kernel": "small weight grid",
    "feature map": "grid of responses",
    "receptive field": "input region seen",
}
terms


#### 4. Step-by-step breakdown

The dictionary maps each term to a plain-English meaning.

These meanings are enough to read most beginner CNN code.

Later chapters add deeper architecture patterns.

The core operation remains local weighted summation.

#### 5. Connection to ML systems

CNN debugging often starts by checking kernel shapes, output shapes, and feature map sizes.

#### 6. Common confusion points

- Do not confuse kernel shape with output shape.
- Do not confuse channels with batch size.
- Manual correlation explains the math but is not efficient.
- Learned kernels remove the need to hand-design every feature.


### 7.2.8 Exercises

#### 1. Intuition

These exercises practice cross-correlation and output-size reasoning.

#### 2. Why this exists

Shape fluency prevents many CNN implementation errors.


#### 3. Examples

Exercise 1: compute a tiny output shape.


In [ ]:
input_height = 5
kernel_height = 3
output_height = input_height - kernel_height + 1
output_height


Exercise 2: inspect Conv2d output shape.


In [ ]:
conv = nn.Conv2d(1, 2, kernel_size=3)
X = torch.zeros(4, 1, 5, 5)
conv(X).shape


#### 4. Step-by-step breakdown

Exercise 1 uses the no-padding, stride-1 formula.

Exercise 2 adds batch size and output channels.

The spatial size shrinks from 5 to 3.

The channel count becomes 2.

#### 5. Connection to ML systems

These calculations appear constantly when stacking CNN layers.

#### 6. Common confusion points

- Batch size does not change because of convolution.
- Output channels come from the number of kernels.
- Spatial size depends on kernel, padding, and stride.
- Always test shapes with tiny tensors.
